# Ingestion — Auto Loader Bronze Layer

This notebook ingests raw CSV files from the Olist Brazilian E-Commerce dataset into Bronze Delta tables using **Databricks Auto Loader**.

**What it does:**
1. Creates Unity Catalog schemas (`bronze`, `silver`, `gold`, `raw`)
2. Creates a UC volume for source files and Auto Loader state
3. Runs Auto Loader for each of the 9 source tables — CSV → Delta
4. All columns land as `StringType` (rescue mode handles schema drift)

**Auto Loader configuration (same for all tables):**
- Format: CSV with header
- Trigger: `availableNow=True` (batch mode — process all pending files, then stop)
- Output mode: `append` (never overwrite, preserves history)
- Schema location: tracks evolving schema per table
- Checkpoint location: ensures exactly-once processing

**Output:** 9 Bronze Delta tables in `ecommerce_data.bronze`

In [0]:
%sql
-- Create all Unity Catalog schemas needed for the medallion architecture.
-- bronze: raw ingested data | silver: cleaned data | gold: analytics-ready data
-- raw: UC volume for source CSV files and Auto Loader state
CREATE SCHEMA IF NOT EXISTS ecommerce_data.bronze;
CREATE SCHEMA IF NOT EXISTS ecommerce_data.silver;
CREATE SCHEMA IF NOT EXISTS ecommerce_data.gold;
CREATE SCHEMA IF NOT EXISTS ecommerce_data.raw;

In [0]:
%sql
-- Create a UC volume to store source CSV files, Auto Loader schema
-- inference state (_schemas/), and streaming checkpoints (_checkpoints/).
create volume if not exists ecommerce_data.raw.raw_files

# Auto Loader — Ingest CSV Files into Bronze

Each cell below ingests one source table. The pattern is identical for all 9 tables:

1. **Read stream** — `spark.readStream.format("cloudFiles")` watches the source directory
2. **Schema inference** — Auto Loader infers schema from file contents; state stored in `_schemas/<table>/`
3. **Write stream** — `df.writeStream.format("delta")` writes to Bronze Delta table
4. **Checkpoint** — `_checkpoints/<table>/` tracks processed files for exactly-once semantics
5. **Trigger** — `availableNow=True` processes all pending files in one batch, then stops

All columns arrive as `StringType` because Auto Loader uses rescue mode. Type conversion happens later in the Silver pipeline.

### orders
Source: `olist_orders_dataset.csv` → `ecommerce_data.bronze.orders`
99,441 rows | PK: `order_id` | Columns: order status, 4 timestamp columns

In [0]:
# ── orders ────────────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/orders/ (CSV)
# Target: ecommerce_data.bronze.orders (Delta)
# Rows: ~99K | PK: order_id
# All columns arrive as StringType from Auto Loader rescue mode.
# ───────────────────────────────────────────────────────────────────────

df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/orders/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/orders/")

df.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/orders/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.orders")

### order_items
Source: `olist_order_items_dataset.csv` → `ecommerce_data.bronze.order_items`
112,650 rows | PK: `order_id` + `order_item_id` | Columns: product, seller, price, freight

In [0]:
# ── order_items ─────────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/order_items/ (CSV)
# Target: ecommerce_data.bronze.order_items (Delta)
# Rows: ~113K | PK: order_id + order_item_id
# All columns arrive as StringType from Auto Loader rescue mode.
# ───────────────────────────────────────────────────────────────────────

df1 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/order_items/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/order_items/")

df1.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/order_items/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.order_items")

### order_payments
Source: `olist_order_payments_dataset.csv` → `ecommerce_data.bronze.order_payments`
103,886 rows | PK: `order_id` + `payment_sequential` | Columns: payment type, installments, value

In [0]:
# ── order_payments ────────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/order_payments/ (CSV)
# Target: ecommerce_data.bronze.order_payments (Delta)
# Rows: ~104K | PK: order_id + payment_sequential
# All columns arrive as StringType from Auto Loader rescue mode.
# ───────────────────────────────────────────────────────────────────────

df2 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/order_payments/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/order_payments/")

df2.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/order_payments/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.order_payments")

### order_reviews
Source: `olist_order_reviews_dataset.csv` → `ecommerce_data.bronze.order_reviews`
104,162 rows | PK: `review_id` | Columns: review score, comment title/message, 2 timestamps

**Note:** Some rows have corrupted `review_score` values (Auto Loader rescue mode mixed comment text into the score column). These are handled in the Silver pipeline.

In [0]:
# ── order_reviews ──────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/order_reviews/ (CSV)
# Target: ecommerce_data.bronze.order_reviews (Delta)
# Rows: ~104K | PK: review_id
# DATA QUALITY NOTE: Some rows have corrupted review_score values
# (Auto Loader rescue mode mixed comment text into the score column).
# These are handled in the Silver pipeline with try_cast().
# ─────────────────────────────────────────────────────────────────────

df3 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/order_reviews/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/order_reviews/")

df3.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/order_reviews/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.order_reviews")

### products
Source: `olist_products_dataset.csv` → `ecommerce_data.bronze.products`
32,951 rows | PK: `product_id` | Columns: category name, dimensions, weight

**Note:** Source CSV has typos in column names (`lenght` instead of `length`). Fixed in the Silver pipeline.

In [0]:
# ── products ────────────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/products/ (CSV)
# Target: ecommerce_data.bronze.products (Delta)
# Rows: ~33K | PK: product_id
# Note: Source CSV has typos in column names (lenght -> length).
# These are fixed in the Silver pipeline select aliases.
# ───────────────────────────────────────────────────────────────────────

df4 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/products/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/products/")

df4.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/products/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.products")

### sellers
Source: `olist_sellers_dataset.csv` → `ecommerce_data.bronze.sellers`
3,095 rows | PK: `seller_id` | Columns: zip code, city, state

In [0]:
# ── sellers ────────────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/sellers/ (CSV)
# Target: ecommerce_data.bronze.sellers (Delta)
# Rows: ~3K | PK: seller_id
# All columns arrive as StringType from Auto Loader rescue mode.
# ───────────────────────────────────────────────────────────────────────

df5 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/sellers/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/sellers/")

df5.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/sellers/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.sellers")

### customers
Source: `olist_customers_dataset.csv` → `ecommerce_data.bronze.customers`
99,441 rows | PK: `customer_id` | Columns: unique customer ID, zip code, city, state

In [0]:
# ── customers ──────────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/customers/ (CSV)
# Target: ecommerce_data.bronze.customers (Delta)
# Rows: ~99K | PK: customer_id
# Note: One customer_id per order; customer_unique_id links repeat customers.
# All columns arrive as StringType from Auto Loader rescue mode.
# ───────────────────────────────────────────────────────────────────────

df6 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/customers/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/customers/")

df6.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/customers/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.customers")

### category_translations
Source: `product_category_name_translation.csv` → `ecommerce_data.bronze.category_translations`
71 rows | PK: `product_category_name` | Columns: Portuguese name, English translation

In [0]:
# ── category_translations ──────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/category_translations/ (CSV)
# Target: ecommerce_data.bronze.category_translations (Delta)
# Rows: 71 | PK: product_category_name
# Maps Portuguese category names to English for dashboard readability.
# ───────────────────────────────────────────────────────────────────────

df7 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/category_translations/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/category_translations/")

df7.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/category_translations/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.category_translations")

### geolocation
Source: `olist_geolocation_dataset.csv` → `ecommerce_data.bronze.geolocation`
1,000,163 rows | No single PK | Columns: zip code, lat, lng, city, state

**Note:** Multiple rows per zip code — each represents a different coordinate within the same postal area. This is legitimate and preserved in Silver.

In [0]:
# ── geolocation ──────────────────────────────────────────────────────
# Source: /Volumes/ecommerce_data/raw/raw_files/geolocation/ (CSV)
# Target: ecommerce_data.bronze.geolocation (Delta)
# Rows: ~1M | No single PK (multiple rows per zip code)
# Each row represents a different coordinate within a postal area.
# This is legitimate — NOT deduplicated by zip code in Silver.
# ───────────────────────────────────────────────────────────────────────

df8 = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation","/Volumes/ecommerce_data/raw/raw_files/_schemas/geolocation/") \
    .option("header", "true") \
    .load("/Volumes/ecommerce_data/raw/raw_files/geolocation/")

df8.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecommerce_data/raw/raw_files/_checkpoints/geolocation/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce_data.bronze.geolocation")